# Bias Detection via Symmetric-Antisymmetric Attention Decomposition

**Research question:** Do stereotypical vs. anti-stereotypical sentences produce measurably different spectral signatures in their attention maps?

**Dataset:** [CrowS-Pairs](https://github.com/nyu-mll/crows-pairs) — 1,508 minimal sentence pairs covering 9 bias categories (gender, race, religion, age, etc.). Each pair differs only by a demographic group.

**Metrics we compare between stereotypical and anti-stereotypical sentences:**
- **Asymmetry score** — ratio of antisymmetric energy to total energy
- **Top eigenvalue** of symmetric component (dominant mutual attention mode)
- **Top eigenvalue** of antisymmetric component (dominant directional flow)
- **Spectral entropy** — how spread vs. concentrated the eigenvalue distribution is

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from attention_steering import AttentionDecomposer, AttentionExtractor
from attention_steering.bias_analysis import (
    load_crows_pairs,
    run_bias_analysis,
    summarize_by_category,
    analyze_pair,
    BIAS_CATEGORIES,
)
from attention_steering.viz import (
    plot_attention_decomposition,
    plot_eigenvalue_spectra,
)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 30)
%matplotlib inline

## 1. Load Model and Dataset

In [ ]:
# GPT-2 (runs on CPU, good for initial exploration)
extractor = AttentionExtractor("gpt2")

# For Qwen2.5-VL on Colab (uncomment):
# extractor = AttentionExtractor("Qwen/Qwen2.5-VL-3B-Instruct", load_in_4bit=True)

print(extractor.get_model_info())

In [ ]:
# Load the CrowS-Pairs dataset
df_pairs = load_crows_pairs()
print(f"Total pairs: {len(df_pairs)}")
print(f"\nBias categories:")
print(df_pairs['bias_type'].value_counts())

In [ ]:
# Preview some pairs
for _, row in df_pairs.head(5).iterrows():
    print(f"[{row['bias_type']}]")
    print(f"  Stereotypical:      {row['sent_more']}")
    print(f"  Anti-stereotypical: {row['sent_less']}")
    print()

## 2. Single Pair Deep Dive

Before running the full analysis, let's look at one pair in detail to build intuition.

In [ ]:
# Pick a gender bias example
gender_pairs = df_pairs[df_pairs['bias_type'] == 'gender'].reset_index(drop=True)
example = gender_pairs.iloc[0]
print(f"Stereotypical:      {example['sent_more']}")
print(f"Anti-stereotypical: {example['sent_less']}")

In [ ]:
decomposer = AttentionDecomposer()

attn_stereo = extractor.extract(example['sent_more'], return_tokens=True)
attn_anti = extractor.extract(example['sent_less'], return_tokens=True)

# Compare asymmetry scores across all layers and heads
layer, head = 5, 0  # pick a mid-layer head

A_s = attn_stereo.get(layer, head)
A_a = attn_anti.get(layer, head)

print(f"Asymmetry score (stereotypical):      {decomposer.asymmetry_score(A_s).item():.4f}")
print(f"Asymmetry score (anti-stereotypical): {decomposer.asymmetry_score(A_a).item():.4f}")

In [ ]:
# Visualize decomposition side by side
fig = plot_attention_decomposition(
    A_s, tokens=attn_stereo.tokens,
    title=f"STEREOTYPICAL — Layer {layer}, Head {head}"
)
plt.show()

fig = plot_attention_decomposition(
    A_a, tokens=attn_anti.tokens,
    title=f"ANTI-STEREOTYPICAL — Layer {layer}, Head {head}"
)
plt.show()

In [ ]:
# Compare eigenvalue spectra
fig = plot_eigenvalue_spectra(A_s, title="Stereotypical — Eigenvalue Spectra")
plt.show()

fig = plot_eigenvalue_spectra(A_a, title="Anti-Stereotypical — Eigenvalue Spectra")
plt.show()

## 3. Full Dataset Analysis

Run the spectral comparison across all pairs (or a subset). This computes asymmetry scores, top eigenvalues, and spectral entropy for every pair.

In [ ]:
# Run analysis — start with a subset for speed, increase for final results
# Use max_pairs=None for the full dataset (1508 pairs, ~30-60 min on GPU)
results_df, raw_results = run_bias_analysis(
    extractor,
    max_pairs=200,  # start small, increase once you've validated the pipeline
)

print(f"Analyzed {len(results_df)} pairs")
results_df.head()

## 4. Asymmetry Score: Stereo vs. Anti-Stereo

The asymmetry score measures directional information flow. If biased processing creates different attention flow patterns, we should see a systematic shift.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Distribution of asymmetry delta
axes[0].hist(results_df['asymmetry_delta'], bins=40, color='#E91E63', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[0].axvline(x=results_df['asymmetry_delta'].mean(), color='blue', linestyle='-', linewidth=2,
                label=f"Mean: {results_df['asymmetry_delta'].mean():.4f}")
axes[0].set_xlabel('Asymmetry Delta (stereo - anti-stereo)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Asymmetry Difference')
axes[0].legend()

# 2. Scatter: stereo vs anti-stereo asymmetry
axes[1].scatter(
    results_df['stereo_asymmetry'], results_df['antistereo_asymmetry'],
    alpha=0.5, c='#2196F3', s=20
)
lims = [0, max(results_df['stereo_asymmetry'].max(), results_df['antistereo_asymmetry'].max()) + 0.01]
axes[1].plot(lims, lims, 'k--', alpha=0.5, label='y=x (no difference)')
axes[1].set_xlabel('Stereotypical Asymmetry')
axes[1].set_ylabel('Anti-Stereotypical Asymmetry')
axes[1].set_title('Asymmetry Score Comparison')
axes[1].legend()

# 3. By bias category
category_means = results_df.groupby('bias_type')['asymmetry_delta'].mean().sort_values()
category_means.plot(kind='barh', ax=axes[2], color='#4CAF50', edgecolor='black')
axes[2].axvline(x=0, color='black', linestyle='--')
axes[2].set_xlabel('Mean Asymmetry Delta')
axes[2].set_title('Asymmetry Delta by Bias Category')

plt.tight_layout()
plt.show()

# Statistical significance
t_stat, p_value = stats.ttest_rel(
    results_df['stereo_asymmetry'],
    results_df['antistereo_asymmetry']
)
print(f"\nPaired t-test (asymmetry): t={t_stat:.4f}, p={p_value:.4f}")
print(f"{'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")

## 5. Eigenvalue Analysis: Symmetric Component

The symmetric component captures mutual attention. Its top eigenvalue reflects the strength of the dominant "agreement" mode.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Top symmetric eigenvalue delta
axes[0].hist(results_df['sym_top_eig_delta'], bins=40, color='#2196F3', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[0].axvline(x=results_df['sym_top_eig_delta'].mean(), color='red', linestyle='-', linewidth=2,
                label=f"Mean: {results_df['sym_top_eig_delta'].mean():.4f}")
axes[0].set_xlabel('Top Symmetric Eigenvalue Delta')
axes[0].set_title('Symmetric λ₁ Difference (stereo - anti)')
axes[0].legend()

# 2. Spectral entropy delta
axes[1].hist(results_df['sym_entropy_delta'], bins=40, color='#FF9800', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[1].axvline(x=results_df['sym_entropy_delta'].mean(), color='red', linestyle='-', linewidth=2,
                label=f"Mean: {results_df['sym_entropy_delta'].mean():.4f}")
axes[1].set_xlabel('Symmetric Spectral Entropy Delta')
axes[1].set_title('Symmetric Spectral Entropy Difference')
axes[1].legend()

# 3. Per-category
category_eig = results_df.groupby('bias_type')['sym_top_eig_delta'].mean().sort_values()
category_eig.plot(kind='barh', ax=axes[2], color='#9C27B0', edgecolor='black')
axes[2].axvline(x=0, color='black', linestyle='--')
axes[2].set_xlabel('Mean Symmetric λ₁ Delta')
axes[2].set_title('Symmetric Top Eigenvalue Delta by Category')

plt.tight_layout()
plt.show()

t_stat, p_value = stats.ttest_rel(
    results_df['stereo_sym_top_eig'],
    results_df['antistereo_sym_top_eig']
)
print(f"Paired t-test (symmetric top eigenvalue): t={t_stat:.4f}, p={p_value:.4f}")
print(f"{'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")

## 6. Eigenvalue Analysis: Antisymmetric Component

The antisymmetric component captures directional flow. Its imaginary eigenvalues represent rotational dynamics in attention.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Top antisymmetric eigenvalue delta
axes[0].hist(results_df['asym_top_eig_delta'], bins=40, color='#4CAF50', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[0].axvline(x=results_df['asym_top_eig_delta'].mean(), color='red', linestyle='-', linewidth=2,
                label=f"Mean: {results_df['asym_top_eig_delta'].mean():.4f}")
axes[0].set_xlabel('Top Antisymmetric Eigenvalue Delta')
axes[0].set_title('Antisymmetric |λ₁| Difference (stereo - anti)')
axes[0].legend()

# 2. Antisymmetric spectral entropy
axes[1].hist(results_df['asym_entropy_delta'], bins=40, color='#F44336', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1.5)
axes[1].axvline(x=results_df['asym_entropy_delta'].mean(), color='blue', linestyle='-', linewidth=2,
                label=f"Mean: {results_df['asym_entropy_delta'].mean():.4f}")
axes[1].set_xlabel('Antisymmetric Spectral Entropy Delta')
axes[1].set_title('Antisymmetric Spectral Entropy Difference')
axes[1].legend()

# 3. Per-category
category_asym_eig = results_df.groupby('bias_type')['asym_top_eig_delta'].mean().sort_values()
category_asym_eig.plot(kind='barh', ax=axes[2], color='#00BCD4', edgecolor='black')
axes[2].axvline(x=0, color='black', linestyle='--')
axes[2].set_xlabel('Mean Antisymmetric |λ₁| Delta')
axes[2].set_title('Antisymmetric Top Eigenvalue Delta by Category')

plt.tight_layout()
plt.show()

t_stat, p_value = stats.ttest_rel(
    results_df['stereo_asym_top_eig'],
    results_df['antistereo_asym_top_eig']
)
print(f"Paired t-test (antisymmetric top eigenvalue): t={t_stat:.4f}, p={p_value:.4f}")
print(f"{'Significant' if p_value < 0.05 else 'Not significant'} at α=0.05")

## 7. Correlation Matrix: All Spectral Deltas

Are the different spectral metrics correlated with each other? This reveals whether bias shows up as a coherent spectral signature.

In [ ]:
delta_cols = [c for c in results_df.columns if 'delta' in c]
corr = results_df[delta_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, ax=ax, square=True,
    xticklabels=[c.replace('_delta', '') for c in delta_cols],
    yticklabels=[c.replace('_delta', '') for c in delta_cols],
)
ax.set_title('Correlation Between Spectral Delta Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Layer-wise Asymmetry Divergence

At which layers does the asymmetry between stereotypical and anti-stereotypical sentences diverge most? This reveals where in the network bias manifests.

In [ ]:
# Aggregate layer-wise asymmetry across all pairs
num_layers = extractor.num_layers
stereo_by_layer = np.zeros(num_layers)
anti_by_layer = np.zeros(num_layers)

for r in raw_results:
    n = min(len(r.stereo_layer_asymmetry), num_layers)
    stereo_by_layer[:n] += r.stereo_layer_asymmetry[:n]
    anti_by_layer[:n] += r.antistereo_layer_asymmetry[:n]

stereo_by_layer /= len(raw_results)
anti_by_layer /= len(raw_results)
delta_by_layer = stereo_by_layer - anti_by_layer

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(range(num_layers), stereo_by_layer, 'o-', label='Stereotypical', color='#E91E63', linewidth=2)
ax1.plot(range(num_layers), anti_by_layer, 's-', label='Anti-stereotypical', color='#2196F3', linewidth=2)
ax1.set_xlabel('Layer')
ax1.set_ylabel('Mean Asymmetry Score')
ax1.set_title('Attention Asymmetry Across Layers')
ax1.legend()
ax1.grid(alpha=0.3)

colors = ['#4CAF50' if d >= 0 else '#F44336' for d in delta_by_layer]
ax2.bar(range(num_layers), delta_by_layer, color=colors, edgecolor='black', alpha=0.8)
ax2.axhline(y=0, color='black', linestyle='--')
ax2.set_xlabel('Layer')
ax2.set_ylabel('Asymmetry Delta (stereo - anti)')
ax2.set_title('Layer-wise Asymmetry Divergence')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

peak_layer = np.argmax(np.abs(delta_by_layer))
print(f"Peak divergence at layer {peak_layer} (delta={delta_by_layer[peak_layer]:.5f})")

## 9. Summary Table by Bias Category

In [ ]:
summary = summarize_by_category(results_df)
print(summary.to_string())

## 10. Effect Size Analysis

Cohen's d for each metric — measures how large the spectral difference is in standard deviation units.

In [ ]:
def cohens_d(x, y):
    """Paired Cohen's d."""
    diff = x - y
    return diff.mean() / (diff.std() + 1e-10)

metrics = {
    'Asymmetry': ('stereo_asymmetry', 'antistereo_asymmetry'),
    'Sym Top λ': ('stereo_sym_top_eig', 'antistereo_sym_top_eig'),
    'Sym Entropy': ('stereo_sym_entropy', 'antistereo_sym_entropy'),
    'Asym Top |λ|': ('stereo_asym_top_eig', 'antistereo_asym_top_eig'),
    'Asym Entropy': ('stereo_asym_entropy', 'antistereo_asym_entropy'),
}

effect_sizes = {}
for name, (col_s, col_a) in metrics.items():
    d = cohens_d(results_df[col_s], results_df[col_a])
    t, p = stats.ttest_rel(results_df[col_s], results_df[col_a])
    effect_sizes[name] = {'Cohen\'s d': d, 't-stat': t, 'p-value': p}

effect_df = pd.DataFrame(effect_sizes).T
effect_df['Significant (p<0.05)'] = effect_df['p-value'] < 0.05
print(effect_df.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#4CAF50' if d >= 0 else '#F44336' for d in effect_df["Cohen's d"]]
effect_df["Cohen's d"].plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linestyle='--')
ax.axvline(x=0.2, color='gray', linestyle=':', alpha=0.5, label='Small effect (0.2)')
ax.axvline(x=-0.2, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel("Cohen's d")
ax.set_title('Effect Size: Stereotypical vs Anti-Stereotypical Spectral Signatures')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Key Findings

Interpret your results here:

- **Asymmetry delta > 0** means stereotypical sentences have more directional (asymmetric) attention flow
- **Symmetric λ₁ delta > 0** means stereotypical sentences have a stronger dominant mutual attention mode  
- **Spectral entropy delta < 0** means stereotypical sentences have more concentrated (less diverse) attention eigenvalues
- **Layer-wise divergence** reveals where in the network the model "decides" to process biased content differently

If significant correlations are found, this opens the door to **bias mitigation via spectral steering** — modifying the eigenvalues at the identified layers to equalize the spectral signatures.